# Agent 2 — Tools: the Model Asks, Your Code Acts

Lesson 1's planner was a script. Today the real model plans: we describe
`search` and `fetch` to it, and it replies with tool requests as JSON.
The dispatch table — the safety line — does not change at all.

**No class API key?** Every live cell has its output precomputed below it;
run the offline cells and read the rest.

In [ ]:
# The mini-web: nine pages about the (fictional) Riverside Community Garden.
# Small enough to read whole, real enough to research. One page is wrong on purpose.
MINIWEB = {
 "riverside-garden.org/about": {"date": "2026-05-10", "title": "Our garden today",
  "text": "The Riverside Community Garden has 60 plots and 48 member families. "
          "We grow vegetables for members and donate surplus to the food pantry."},
 "riverside-garden.org/history": {"date": "2023-05-02", "title": "Our history",
  "text": "Founded in 2019 with a dozen beds. The sign by the gate lists 48 plots, "
          "painted when we finished the 2023 season."},
 "riverside-garden.org/join": {"date": "2026-06-01", "title": "Join us",
  "text": "Want a plot? The waitlist currently holds 22 families. Members pay a "
          "small annual fee and share watering duties."},
 "lakeview-news.com/garden-expands": {"date": "2026-04-20", "title": "Garden adds 12 plots",
  "text": "The Riverside Community Garden completed its expansion this spring, "
          "taking the garden from 48 plots to 60. Organizers credit a city grant."},
 "lakeview-news.com/roundup-2023": {"date": "2023-09-15", "title": "Community roundup",
  "text": "At the Riverside garden, 31 member families closed out the 2023 season "
          "with a harvest festival."},
 "cityparks.gov/report-2026": {"date": "2026-03-14", "title": "Community garden census",
  "text": "Riverside Community Garden: 60 plots, 48 member families, established "
          "2019. Census conducted March 2026."},
 "cityparks.gov/grants-2025": {"date": "2025-11-08", "title": "2025 grant awards",
  "text": "Riverside Community Garden: $15,000 for expansion. The site's land "
          "lease with the parks department runs through 2028."},
 "gardenblog.example.com/visit": {"date": "2026-02-02", "title": "A visit to Riverside",
  "text": "Lovely afternoon at Riverside! I heard they have 600 plots now, which "
          "explains the crowds. The tomatoes were spectacular."},
 "gardenblog.example.com/opinion": {"date": "2026-01-05", "title": "Why gardens matter",
  "text": "Community gardens are the beating heart of a neighborhood. Riverside "
          "is a treasure and everyone loves it."},
}

import re as _re, collections as _c
def _words(text):
    return set(w for w in _re.findall(r"[a-z0-9]+", text.lower()) if len(w) > 2)
_DF = _c.Counter()                       # in how many pages does each word appear?
for _p in MINIWEB.values():
    for _w in _words(_p["title"] + " " + _p["text"]):
        _DF[_w] += 1

def search(query):
    """Score pages by shared words, each weighted by rarity (1/pages-containing-it).
    'riverside' is on every page and says nothing; 'waitlist' is on one and says a lot."""
    qwords = _words(query)
    scored = []
    for url, page in MINIWEB.items():
        shared = qwords & _words(page["title"] + " " + page["text"])
        scored.append((sum(1.0 / _DF[w] for w in shared), url, page["title"]))
    scored.sort(reverse=True)
    return [(url, title) for score, url, title in scored[:3] if score > 0.3]

def fetch(url):
    """Return a page's text with its receipt (url and date) attached."""
    page = MINIWEB[url]
    return {"url": url, "date": page["date"], "text": page["text"]}

print(f"{len(MINIWEB)} pages online.")
print("search('riverside garden plots') ->")
for url, title in search("riverside garden plots"):
    print("  ", url, "-", title)

In [ ]:
%pip install -q anthropic

In [ ]:
import os, getpass
# Ask your teacher for the class API key. It is never typed into a cell,
# never saved in the notebook - getpass keeps it out of your file.
try:
    os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Class API key: ")
    HAVE_KEY = len(os.environ["ANTHROPIC_API_KEY"]) > 10
except Exception:
    HAVE_KEY = False
print("Key loaded." if HAVE_KEY else "No key - the notebook still teaches: precomputed outputs are shown below each live cell.")

In [ ]:
MODEL = "claude-opus-5"

def ask(prompt, system=None, max_tokens=1000):
    """One model call, plain text in and out."""
    import anthropic
    client = anthropic.Anthropic()
    kwargs = dict(model=MODEL, max_tokens=max_tokens,
                  messages=[{"role": "user", "content": prompt}])
    if system:
        kwargs["system"] = system
    return client.messages.create(**kwargs).content[-1].text

def get_json(prompt, tries=3):
    """Ask for JSON only; parse; re-ask on failure. The retry pattern from Build with LLMs."""
    import json as _json
    for attempt in range(tries):
        text = ask(prompt + "\n\nReply with ONLY valid JSON.")
        try:
            start = text.index("[") if "[" in text.split("{")[0] else text.index("{")
            return _json.loads(text[start:])
        except (ValueError, KeyError):
            continue
    raise RuntimeError("no valid JSON after retries")

## Describing the tools

A tool description is plain language: name, what it does, arguments. The
model reads these and decides when to use them. (The Claude API also has a
structured `tools=` parameter that does this formally — same idea, more
plumbing; we use the JSON-in-text form so every part stays visible.)

In [ ]:
TOOL_PROMPT = """You are a research agent. You can request actions by replying
with ONLY a JSON object, no other text:
  {"tool": "search", "query": "<words to search>"}   - search the web, returns urls
  {"tool": "fetch", "url": "<url>"}                  - read one page
  {"tool": "answer", "text": "<final answer with [source urls]>"}  - finish

Rules: cite the url of any page a fact came from. Prefer official sources.
Question: """

def parse_action(reply):
    import json as _json
    start = reply.index("{")
    return _json.loads(reply[start:reply.rindex("}") + 1])

# The loop, with the model planning. Same dispatch as lesson 1.
def run_agent(question, max_turns=6):
    conversation = TOOL_PROMPT + question
    for turn in range(max_turns):
        if HAVE_KEY:
            reply = ask(conversation)
        else:
            reply = PRECOMPUTED_TURNS[turn]          # replay of a real-shaped run
        act = parse_action(reply)
        print(f"MODEL: {act}")
        if act["tool"] == "answer":
            return act["text"]
        if act["tool"] == "search":
            result = str(search(act["query"]))
        elif act["tool"] == "fetch":
            result = str(fetch(act["url"]))
        else:
            result = f"ERROR: no tool named {act['tool']!r}"
        print(f"CODE:  {result}")
        print()
        conversation += f"\n{reply}\nRESULT: {result}\n"
    return "(ran out of turns)"

PRECOMPUTED_TURNS = [
 '{"tool": "search", "query": "Riverside Community Garden member families"}',
 '{"tool": "fetch", "url": "cityparks.gov/report-2026"}',
 '{"tool": "answer", "text": "The garden has 48 member families, per the March 2026 city census [cityparks.gov/report-2026]."}',
]

answer = run_agent("How many member families does the Riverside garden have?")
print("FINAL:", answer)

The precomputed run above replays the same three turns a live run
produces. With the class key, delete `PRECOMPUTED_TURNS` from the loop and
watch the model plan for itself — including, sometimes, a different but
equally good route (the about page also holds the number).

## The dispatch table is the boundary

Ask the agent to do something it has no hands for:

In [ ]:
# The model can REQUEST anything. Only the dispatch table decides what RUNS.
bad = '{"tool": "send_email", "to": "everyone@school.org"}'
act = parse_action(bad)
result = f"ERROR: no tool named {act['tool']!r}" if act["tool"] not in ("search", "fetch", "answer") else "would run"
print("model requested:", act)
print("dispatch says:  ", result)

## Try it

1. Add a `calculator` tool to the dispatch (hint: a function that calls
   `eval` is the easy version — write one sentence on why professionals
   don't ship `eval` on untrusted input).
2. Ask a question the mini-web can't answer ("what's the garden's budget?")
   and watch what the model does with empty search results.
3. **Build turn-in:** the tool set you designed for someone you know — each
   tool's name, one job sentence, arguments — plus the tool you deliberately
   withheld, and why.